In [2]:
# ================================================================
# 03_BASELINE_MODEL.PY
# Anticipatory Solar Forecasting Research
# Reproduce the original paper's Random Forest exactly
# This becomes Layer 1 of our two-layer system
# ================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import os

os.makedirs("models", exist_ok=True)
os.makedirs("results/baseline", exist_ok=True)

# ----------------------------------------------------------------
# 1. EXACT FEATURES FROM THE ORIGINAL PAPER
# ----------------------------------------------------------------
# We use ONLY these features for the baseline
# so our results are directly comparable to theirs
BASELINE_FEATURES = [
    "hour",
    "cloud_cover",
    "hour_x_cloud",
    "temp_x_humidity",
    "day_of_year",
]

TARGET = "GHI"

# ----------------------------------------------------------------
# 2. EXACT MODEL PARAMETERS FROM THE ORIGINAL PAPER
# ----------------------------------------------------------------
RF_PARAMS = {
    "n_estimators":    200,
    "min_samples_leaf": 5,
    "max_features":    "sqrt",
    "random_state":    42,
    "n_jobs":          -1,   # use all CPU cores
}

# ----------------------------------------------------------------
# 3. TRAIN + EVALUATE FUNCTION
# ----------------------------------------------------------------
def train_baseline(city_name):
    print(f"\n{'='*55}")
    print(f"  Baseline Model: {city_name}")
    print(f"{'='*55}")

    # --- Load processed data ---
    df = pd.read_csv(
        f"data/processed/{city_name.lower()}_processed.csv"
    )

    # --- Split by year ---
    train = df[df["split"] == "train"]
    val   = df[df["split"] == "val"]
    test  = df[df["split"] == "test"]

    X_train = train[BASELINE_FEATURES]
    y_train = train[TARGET]

    X_val   = val[BASELINE_FEATURES]
    y_val   = val[TARGET]

    X_test  = test[BASELINE_FEATURES]
    y_test  = test[TARGET]

    print(f"  Train: {len(X_train):,} | "
          f"Val: {len(X_val):,} | "
          f"Test: {len(X_test):,}")

    # --- Train model ---
    print(f"  Training Random Forest...")
    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train, y_train)
    print(f"  ✓ Training complete")

    # --- Generate predictions ---
    pred_train = model.predict(X_train)
    pred_val   = model.predict(X_val)
    pred_test  = model.predict(X_test)

    # --- Compute metrics ---
    def metrics(y_true, y_pred, split_name):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        bias = np.mean(y_pred - y_true)
        return {
            "city":  city_name,
            "split": split_name,
            "RMSE":  round(rmse, 2),
            "MAE":   round(mae, 2),
            "R2":    round(r2, 4),
            "Bias":  round(bias, 2),
        }

    results = [
        metrics(y_train, pred_train, "train"),
        metrics(y_val,   pred_val,   "val"),
        metrics(y_test,  pred_test,  "test"),
    ]

    # --- Print results ---
    print(f"\n  {'Split':<8} {'RMSE':>8} {'MAE':>8} "
          f"{'R2':>8} {'Bias':>8}")
    print(f"  {'-'*40}")
    for r in results:
        print(f"  {r['split']:<8} {r['RMSE']:>8} {r['MAE']:>8} "
              f"{r['R2']:>8} {r['Bias']:>8}")

    # --- Compare to paper ---
    paper_rmse = {
        "Riyadh": 100.6, "Jeddah": 101.0,
        "Abha": 107.4, "Dammam": 92.4, "Qassim": 97.7
    }
    test_rmse = results[2]["RMSE"]
    paper_val = paper_rmse[city_name]
    diff      = abs(test_rmse - paper_val)

    print(f"\n  Paper RMSE:  {paper_val}")
    print(f"  Our RMSE:    {test_rmse}")
    print(f"  Difference:  {diff:.1f} W/m²  ", end="")

    if diff < 15:
        print("✓ Consistent with paper")
    else:
        print("⚠ Larger difference — expected with more data")

    # --- Save residuals for meta-model (CRITICAL) ---
    # These residuals ARE the meta-model's training labels
    # We save residuals for ALL splits
    full_X = df[BASELINE_FEATURES]
    full_pred = model.predict(full_X)
    df["GHI_predicted"]  = full_pred
    df["residual"]       = np.abs(df[TARGET] - full_pred)
    df["signed_residual"] = df[TARGET] - full_pred

    residual_path = (
        f"results/baseline/{city_name.lower()}_residuals.csv"
    )
    df.to_csv(residual_path, index=False)
    print(f"\n  ✓ Residuals saved: {residual_path}")

    # --- Save model ---
    model_path = f"models/{city_name.lower()}_baseline_rf.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"  ✓ Model saved: {model_path}")

    return model, results, df


# ----------------------------------------------------------------
# 4. RUN ALL CITIES
# ----------------------------------------------------------------
all_results = []
all_models  = {}
all_dfs     = {}

for city in ["Riyadh", "Jeddah", "Abha", "Dammam", "Qassim"]:
    model, results, df = train_baseline(city)
    all_results.extend(results)
    all_models[city]   = model
    all_dfs[city]      = df


# ----------------------------------------------------------------
# 5. FINAL SUMMARY TABLE
# ----------------------------------------------------------------
results_df = pd.DataFrame(all_results)
test_results = results_df[results_df["split"] == "test"]

print(f"\n{'='*55}")
print("  BASELINE RESULTS SUMMARY (TEST SET 2024)")
print(f"{'='*55}")
print(f"\n  {'City':<10} {'RMSE':>8} {'MAE':>8} "
      f"{'R2':>8} {'Bias':>8}")
print(f"  {'-'*44}")

for _, row in test_results.iterrows():
    print(f"  {row['city']:<10} {row['RMSE']:>8} "
          f"{row['MAE']:>8} {row['R2']:>8} {row['Bias']:>8}")

# Save results table
results_df.to_csv("results/baseline/all_baseline_results.csv",
                  index=False)
print(f"\n  ✓ Results saved to results/baseline/")
print(f"\n  ✓ BASELINE COMPLETE — READY FOR META-MODEL")


  Baseline Model: Riyadh
  Train: 21,915 | Val: 5,475 | Test: 5,490
  Training Random Forest...
  ✓ Training complete

  Split        RMSE      MAE       R2     Bias
  ----------------------------------------
  train       33.17    17.45   0.9897     0.17
  val         49.18    26.73   0.9773    -6.96
  test        51.68    25.57   0.9747    -4.04

  Paper RMSE:  100.6
  Our RMSE:    51.68
  Difference:  48.9 W/m²  ⚠ Larger difference — expected with more data

  ✓ Residuals saved: results/baseline/riyadh_residuals.csv
  ✓ Model saved: models/riyadh_baseline_rf.pkl

  Baseline Model: Jeddah
  Train: 21,915 | Val: 5,475 | Test: 5,490
  Training Random Forest...
  ✓ Training complete

  Split        RMSE      MAE       R2     Bias
  ----------------------------------------
  train       31.91    16.66   0.9906     0.05
  val         48.53    26.19   0.9779    -4.72
  test        49.49    25.93   0.9769     -6.2

  Paper RMSE:  101.0
  Our RMSE:    49.49
  Difference:  51.5 W/m²  ⚠ Large